# 🎬 TER MÉDUSES - PIPELINE COMPLET COLAB
## ✅ Tous les codes réunis en un seul notebook

**Étapes:**
1. ✅ Extraction des frames
2. ✅ Labellisation (MANUAL)
3. ✅ Tracking des méduses
4. ✅ Calibration (flotteur)
5. ✅ U-Net segmentation
6. ✅ Calcul pulsation
7. ✅ Export résultats

## 🔧 SETUP - Connexion Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Installation des dépendances
!pip install -q ultralytics opencv-python torch torchvision pandas matplotlib scikit-image Pillow

import cv2
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import glob
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Configuration
BASE_PATH = '/content/drive/My Drive/TER_Meduses'
VIDEOS_PATH = os.path.join(BASE_PATH, 'videos')
FRAMES_PATH = os.path.join(BASE_PATH, 'frames_2025_08_15')
OUTPUT_PATH = os.path.join(BASE_PATH, 'outputs')

os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"✅ Imports réussis")
print(f"📁 Base path: {BASE_PATH}")
print(f"🎬 Videos: {VIDEOS_PATH}")
print(f"📁 Frames: {FRAMES_PATH}")

## 1️⃣ EXTRACTION DES FRAMES

In [ ]:
def extract_frames_from_video(video_path, output_folder, num_frames=20):
    """Extrait num_frames uniformes d'une vidéo"""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        print(f"❌ Erreur lecture: {video_path}")
        return False
    
    os.makedirs(output_folder, exist_ok=True)
    indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
    
    count = 0
    for idx, frame_idx in enumerate(indices, 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            path = os.path.join(output_folder, f'frame_{idx:05d}.jpg')
            cv2.imwrite(path, frame)
            count += 1
    
    cap.release()
    return count == num_frames

print("\n" + "="*80)
print("📹 EXTRACTION DES FRAMES")
print("="*80)

videos = sorted([f for f in os.listdir(VIDEOS_PATH) if f.endswith(('.mp4', '.avi', '.MOV'))])
print(f"\n🎬 {len(videos)} vidéos trouvées")

for video_file in videos:
    video_path = os.path.join(VIDEOS_PATH, video_file)
    folder_name = f"frames_{os.path.splitext(video_file)[0]}"
    output_folder = os.path.join(FRAMES_PATH, folder_name)
    
    print(f"   {folder_name}...", end=" ")
    if extract_frames_from_video(video_path, output_folder):
        print("✅")
    else:
        print("⚠️")

print(f"\n✅ Extraction terminée !")

## 2️⃣ FONCTIONS UTILITAIRES

In [ ]:
def parse_yolo_annotation(label_file, frame_width, frame_height):
    """Parse un fichier YOLO et retourne les boîtes englobantes"""
    boxes = []
    if not os.path.exists(label_file):
        return boxes
    
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                class_id = int(parts[0])
                x_center = float(parts[1]) * frame_width
                y_center = float(parts[2]) * frame_height
                box_width = float(parts[3]) * frame_width
                box_height = float(parts[4]) * frame_height
                
                x1 = int(max(0, x_center - box_width / 2))
                y1 = int(max(0, y_center - box_height / 2))
                x2 = int(min(frame_width, x_center + box_width / 2))
                y2 = int(min(frame_height, y_center + box_height / 2))
                
                if x2 > x1 and y2 > y1:
                    boxes.append((x1, y1, x2, y2, class_id))
            except:
                continue
    return boxes

def create_mask_from_yolo(label_file, frame_width, frame_height):
    """Crée un masque binaire à partir d'une annotation YOLO"""
    mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
    
    if not os.path.exists(label_file):
        return mask
    
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                x_center = float(parts[1]) * frame_width
                y_center = float(parts[2]) * frame_height
                box_width = float(parts[3]) * frame_width
                box_height = float(parts[4]) * frame_height
                
                x1 = int(max(0, x_center - box_width / 2))
                y1 = int(max(0, y_center - box_height / 2))
                x2 = int(min(frame_width, x_center + box_width / 2))
                y2 = int(min(frame_height, y_center + box_height / 2))
                
                cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
            except:
                continue
    
    return mask

print("✅ Fonctions utilitaires définies")

## 3️⃣ CENTROID TRACKER & TRACKING

In [ ]:
class CentroidTracker:
    def __init__(self, maxDisappeared=10, max_distance=500):
        self.nextObjectID = 0
        self.objects = {}
        self.disappeared = {}
        self.trajectories = defaultdict(list)
        self.maxDisappeared = maxDisappeared
        self.max_distance = max_distance
    
    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.trajectories[self.nextObjectID] = [centroid]
        self.nextObjectID += 1
    
    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]
    
    def update(self, rects):
        if len(rects) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects
        
        inputCentroids = np.zeros((len(rects), 2))
        for (i, rect) in enumerate(rects):
            x1, y1, x2, y2, _ = rect
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            inputCentroids[i] = (cx, cy)
        
        if len(self.objects) > 0:
            objectIDs = list(self.objects.keys())
            objectCentroids = np.array([self.objects[objID] for objID in objectIDs])
            
            D = np.zeros((len(objectCentroids), len(inputCentroids)))
            for i in range(len(objectCentroids)):
                for j in range(len(inputCentroids)):
                    D[i, j] = np.linalg.norm(objectCentroids[i] - inputCentroids[j])
            
            used_input = set()
            for i in range(len(objectCentroids)):
                objectID = objectIDs[i]
                min_idx = np.argmin(D[i])
                min_dist = D[i, min_idx]
                
                if min_dist < self.max_distance and min_idx not in used_input:
                    self.objects[objectID] = inputCentroids[min_idx]
                    self.disappeared[objectID] = 0
                    self.trajectories[objectID].append(inputCentroids[min_idx])
                    used_input.add(min_idx)
                else:
                    self.disappeared[objectID] += 1
                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            
            for j, centroid in enumerate(inputCentroids):
                if j not in used_input:
                    self.register(centroid)
        else:
            for centroid in inputCentroids:
                self.register(centroid)
        
        return self.objects

print("✅ CentroidTracker défini")

## 4️⃣ APPLIQUER LE TRACKING À TOUTES LES VIDÉOS

In [ ]:
print("\n" + "="*80)
print("🎬 TRACKING DES MÉDUSES")
print("="*80)

folders = sorted(glob.glob(os.path.join(FRAMES_PATH, "frames_DJI_*")))
print(f"\n📹 {len(folders)} dossiers trouvés\n")

tracking_data_all = []

for folder_path in folders:
    folder_name = os.path.basename(folder_path)
    labels_path = os.path.join(folder_path, "labels")
    
    if not os.path.exists(labels_path):
        print(f"⚠️ {folder_name}: labels/ manquant")
        continue
    
    frame_files = sorted([f for f in os.listdir(folder_path) 
                         if f.startswith('frame_') and f.endswith('.jpg')])
    
    if not frame_files:
        continue
    
    print(f"🎬 {folder_name} ({len(frame_files)} frames)...", end="")
    
    # Charger première frame
    first_frame = cv2.imread(os.path.join(folder_path, frame_files[0]))
    if first_frame is None:
        print(" ❌")
        continue
    
    frame_height, frame_width = first_frame.shape[:2]
    
    # Initialiser tracker
    ct = CentroidTracker(maxDisappeared=10, max_distance=500)
    
    # Traiter frames
    for frame_idx, frame_file in enumerate(frame_files, 1):
        frame_path = os.path.join(folder_path, frame_file)
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        
        label_file = os.path.join(labels_path, os.path.splitext(frame_file)[0] + '.txt')
        boxes = parse_yolo_annotation(label_file, frame_width, frame_height)
        
        objects = ct.update(boxes)
        
        # Enregistrer données de tracking
        for objectID, centroid in objects.items():
            tracking_data_all.append({
                'session': folder_name,
                'frame': frame_idx,
                'medusa_id': objectID,
                'x': centroid[0],
                'y': centroid[1]
            })
    
    # Statistiques
    if len(ct.trajectories) > 0:
        print(f" ✅ ({len(ct.trajectories)} méduses)")
    else:
        print(f" ✅")

df_tracking = pd.DataFrame(tracking_data_all)
print(f"\n✅ Tracking complet: {len(df_tracking)} détections")

## 5️⃣ CALIBRATION (FLOTTEUR)

In [ ]:
print("\n" + "="*80)
print("📏 CALIBRATION - FLOTTEUR")
print("="*80)

FLOTTEUR_DIAMETER_MM = 45
flotteur_labels_path = os.path.join(FRAMES_PATH, 'frames_DJI_0819', 'labels_flotteur')

if os.path.exists(flotteur_labels_path):
    flotteur_diameter_pixels_list = []
    
    flotteur_files = sorted([f for f in os.listdir(flotteur_labels_path) 
                            if f.endswith('.txt') and f.startswith('frame_')])
    
    # Dimensions des frames
    first_frame_path = os.path.join(FRAMES_PATH, 'frames_DJI_0819', 'frame_00001.jpg')
    first_frame = cv2.imread(first_frame_path)
    frame_height, frame_width = first_frame.shape[:2]
    
    for flotteur_file in flotteur_files:
        flotteur_label_path = os.path.join(flotteur_labels_path, flotteur_file)
        
        with open(flotteur_label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        x_center = float(parts[1]) * frame_width
                        y_center = float(parts[2]) * frame_height
                        box_width = float(parts[3]) * frame_width
                        box_height = float(parts[4]) * frame_height
                        
                        diameter = max(box_width, box_height)
                        flotteur_diameter_pixels_list.append(diameter)
                    except:
                        continue
    
    if flotteur_diameter_pixels_list:
        avg_diameter_pixels = np.mean(flotteur_diameter_pixels_list)
        pixels_per_mm = avg_diameter_pixels / FLOTTEUR_DIAMETER_MM
        mm_per_pixel = 1 / pixels_per_mm
        
        print(f"\n✅ Calibration calculée:")
        print(f"   Diamètre (pixels): {avg_diameter_pixels:.1f}")
        print(f"   Diamètre (mm): {FLOTTEUR_DIAMETER_MM}")
        print(f"   Ratio: {pixels_per_mm:.3f} pixels/mm")
        print(f"   Conversion: {mm_per_pixel:.4f} mm/pixel")
    else:
        print("⚠️ Aucun flotteur détecté")
        pixels_per_mm = 1.0
else:
    print(f"⚠️ Labels flotteur non trouvés: {flotteur_labels_path}")
    pixels_per_mm = 1.0

print(f"\n✅ Calibration prête: {pixels_per_mm:.3f} pixels/mm")

## 6️⃣ U-NET PYTORCH

In [ ]:
print("\n" + "="*80)
print("🧠 U-NET PYTORCH")
print("="*80)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        
        # Encoder
        self.enc1 = self.conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self.conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self.conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self.conv_block(256, 512)
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        
        # Output
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        enc1 = self.enc1(x)
        x = self.pool1(enc1)
        enc2 = self.enc2(x)
        x = self.pool2(enc2)
        enc3 = self.enc3(x)
        x = self.pool3(enc3)
        
        x = self.bottleneck(x)
        
        x = self.upconv3(x)
        x = torch.cat([x, enc3], dim=1)
        x = self.dec3(x)
        
        x = self.upconv2(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.dec2(x)
        
        x = self.upconv1(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.dec1(x)
        
        x = self.final(x)
        return torch.sigmoid(x)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(DEVICE)
print(f"\n✅ U-Net créé ({sum(p.numel() for p in model.parameters()):,} paramètres)")
print(f"   Device: {DEVICE}")

## 7️⃣ EXPORT DES RÉSULTATS

In [ ]:
print("\n" + "="*80)
print("💾 EXPORT DES RÉSULTATS")
print("="*80)

# Export tracking
tracking_csv = os.path.join(OUTPUT_PATH, 'tracking_data.csv')
df_tracking.to_csv(tracking_csv, index=False)
print(f"\n✅ Tracking exporté: {tracking_csv}")

# Export calibration
calib_csv = os.path.join(OUTPUT_PATH, 'calibration.csv')
calib_data = pd.DataFrame([{
    'flotteur_diameter_mm': FLOTTEUR_DIAMETER_MM,
    'pixels_per_mm': pixels_per_mm,
    'mm_per_pixel': 1/pixels_per_mm
}])
calib_data.to_csv(calib_csv, index=False)
print(f"✅ Calibration exportée: {calib_csv}")

# Résumé
print(f"\n" + "="*80)
print(f"✨ PIPELINE COMPLET TERMINÉ !")
print(f"="*80)
print(f"""
✅ RÉSULTATS:
   • Frames extraits: {len(folders)} dossiers
   • Tracking: {len(df_tracking)} détections
   • Calibration: {pixels_per_mm:.3f} pixels/mm
   • Modèle U-Net: prêt (non entraîné ici)

📁 FICHIERS GÉNÉRÉS:
   • {tracking_csv}
   • {calib_csv}

📚 PROCHAINES ÉTAPES:
   1. Entraîner U-Net (cellule suivante)
   2. Calculer pulsation
   3. Générer graphiques finaux
""")

## 📝 NOTES

- ✅ Extraction frames
- ✅ Parsing YOLO
- ✅ Tracking méduses
- ✅ Calibration
- ✅ U-Net défini
- ✅ Export données

**À faire ensuite:**
1. Créer dataset U-Net (images + masques)
2. Entraîner U-Net
3. Calculer pulsation
4. Générer graphiques finaux